# Sanskrit ASR Training
Fine-tuning `facebook/wav2vec2-large-xlsr-53` on the `ai4bharat/Kathbath` Sanskrit dataset using CTC.

## Step 0 — Authenticate with Hugging Face
Run this cell once to log in. The `ai4bharat/Kathbath` dataset is gated and requires approval:
https://huggingface.co/datasets/ai4bharat/Kathbath

In [2]:
from huggingface_hub import login
login()  # Paste your HF token when prompted

ImportError: The `notebook_login` function can only be used in a notebook (Jupyter or Colab) and you need the `ipywidgets` module: `pip install ipywidgets`.

## Step 1 — Imports & Environment

In [ ]:
import os
import re
import json
import torch
import jiwer
import numpy as np
from dataclasses import dataclass
from typing import Union

from datasets import load_dataset, Audio, DatasetDict
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    TrainingArguments,
    Trainer
)
from huggingface_hub import get_token

# Fix PyTorch CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME = "facebook/wav2vec2-large-xlsr-53"
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Step 2 — Load Dataset

In [ ]:
hf_token = get_token()
if hf_token is None:
    raise RuntimeError("No Hugging Face token found. Run the login() cell above first.")

dataset = load_dataset("ai4bharat/Kathbath", "sanskrit", token=hf_token)

# Create validation split if missing
if "validation" not in dataset:
    split = dataset["train"].train_test_split(test_size=0.1, seed=42)
    dataset = DatasetDict({
        "train": split["train"],
        "validation": split["test"]
    })

# Reduce dataset size for memory safety (use 50%)
dataset["train"] = dataset["train"].select(range(len(dataset["train"]) // 2))
dataset["validation"] = dataset["validation"].select(range(len(dataset["validation"]) // 2))

print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print(dataset)

## Step 3 — Audio Processing
Rename the audio column, cast it to 16 kHz, and extract the raw waveform array.

In [ ]:
dataset = dataset.rename_column("audio_filepath", "audio")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

def prepare_audio(batch):
    audio = batch["audio"]
    batch["speech"] = audio["array"]
    batch["sampling_rate"] = audio["sampling_rate"]
    return batch

columns_to_remove = [col for col in dataset["train"].column_names if col not in ("text", "speech", "sampling_rate")]
dataset = dataset.map(prepare_audio, remove_columns=columns_to_remove)
print(dataset)

## Step 4 — Text Normalization
Keep only Devanagari characters (U+0900–U+097F) and normalize whitespace.

In [ ]:
def normalize(text):
    text = re.sub(r"[^\u0900-\u097F ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

dataset = dataset.map(lambda x: {"text": normalize(x["text"])})
print("Sample text:", dataset["train"][0]["text"])

## Step 5 — Build Vocabulary & Processor
Extract unique Devanagari characters, build `vocab.json`, and create the `Wav2Vec2Processor`.

In [ ]:
def extract_chars(batch):
    all_text = " ".join(batch["text"])
    vocab = list(set(all_text))
    return {"vocab": [vocab]}

vocab_train = dataset["train"].map(
    extract_chars, batched=True,
    remove_columns=dataset["train"].column_names
)
vocab_val = dataset["validation"].map(
    extract_chars, batched=True,
    remove_columns=dataset["validation"].column_names
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_val["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(vocab_list)}

vocab_dict["|"] = vocab_dict[" "]  # Space → word delimiter
del vocab_dict[" "]
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

print(f"Vocabulary size: {len(vocab_dict)}")

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

print("Processor ready. Vocab size:", len(processor.tokenizer))

## Step 6 — Prepare Dataset (Features + Labels)

In [ ]:
def prepare_dataset(batch):
    # Encode audio
    inputs = processor(batch["speech"], sampling_rate=16000)
    batch["input_values"] = inputs.input_values[0]
    # Encode text labels (use processor(text=...) — NOT the deprecated as_target_processor)
    batch["labels"] = processor(text=batch["text"]).input_ids
    return batch

dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    batched=False
)
print("Dataset prepared:", dataset)

## Step 7 — Load Model

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    vocab_size=len(processor.tokenizer),
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    ignore_mismatched_sizes=True,  # Required when vocab_size differs from checkpoint
)

# Freeze CNN feature encoder — only fine-tune the transformer layers
model.freeze_feature_encoder()

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,} | Trainable: {trainable:,}")

## Step 8 — Data Collator
Pads audio inputs and labels to the longest sequence in each batch.

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        # Replace padding token id with -100 so loss ignores padding
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor)
print("Data collator ready.")

## Step 9 — Metrics (WER)

In [ ]:
def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred_str = processor.batch_decode(pred_ids)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = jiwer.wer(label_str, pred_str)
    return {"wer": wer}

## Step 10 — Training Arguments & Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir="./sanskrit_asr",
    group_by_length=True,              # Batch similar-length sequences together → faster
    per_device_train_batch_size=1,     # Keep low to avoid OOM on GPU
    gradient_accumulation_steps=8,     # Effective batch size = 8
    eval_strategy="steps",
    num_train_epochs=10,
    fp16=True,
    gradient_checkpointing=True,       # Trade compute for memory
    learning_rate=3e-4,
    warmup_steps=500,
    max_grad_norm=1.0,
    logging_steps=100,
    save_steps=1000,
    eval_steps=1000,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=processor.feature_extractor,  # Replaces deprecated tokenizer=
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready. Starting training...")

## Step 11 — Train

In [ ]:
trainer.train()

## Step 12 — Save Model & Processor

In [ ]:
model.save_pretrained("./sanskrit_asr_model")
processor.save_pretrained("./sanskrit_asr_model")
print("Model and processor saved to ./sanskrit_asr_model")

## Step 13 — Evaluation Plots
This section generates alignment, confusion, curves, RTF tradeoffs, and robustness plots from a validation subset.
Phoneme confusion is approximated with character-level confusion (no G2P mapping).
The attention plot is a CTC logit alignment heatmap (not a true attention map).

In [1]:
import os
import time
import math
import json
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch
from jiwer import wer
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

try:
    from sklearn.metrics import confusion_matrix
except ImportError as exc:
    raise ImportError("Please install scikit-learn: pip install scikit-learn") from exc

# ------------------------------
# Config
# ------------------------------
SUBSET_SIZE = 50
TOP_WORDS = 25
SNR_LEVELS_DB = ["clean", 20, 10, 0]
SEED = 42
REPORT_DIR = "reports/figures"
os.makedirs(REPORT_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------
# Load processor/model if needed
# ------------------------------
def _latest_checkpoint(base_dir="./sanskrit_asr"):
    paths = glob.glob(os.path.join(base_dir, "checkpoint-*"))
    if not paths:
        return None
    def _step(p):
        try:
            return int(p.rsplit("-", 1)[-1])
        except Exception:
            return -1
    return sorted(paths, key=_step)[-1]

if "processor" not in globals():
    if os.path.isdir("./sanskrit_asr_model"):
        processor = Wav2Vec2Processor.from_pretrained("./sanskrit_asr_model")
    else:
        ckpt = _latest_checkpoint()
        processor = Wav2Vec2Processor.from_pretrained(ckpt) if ckpt else None
        if processor is None:
            raise RuntimeError("Processor not found. Run training cells or ensure ./sanskrit_asr_model exists.")

if "model" not in globals():
    if os.path.isdir("./sanskrit_asr_model"):
        model = Wav2Vec2ForCTC.from_pretrained("./sanskrit_asr_model")
    else:
        ckpt = _latest_checkpoint()
        if ckpt is None:
            raise RuntimeError("Model checkpoint not found.")
        model = Wav2Vec2ForCTC.from_pretrained(ckpt)

model = model.to(device).eval()

# ------------------------------
# Dataset subset
# ------------------------------
if "dataset" not in globals():
    raise RuntimeError("Dataset not found. Run the dataset loading/prep cells first.")

val_ds = dataset["validation"]
subset_size = min(SUBSET_SIZE, len(val_ds))
val_ds = val_ds.select(range(subset_size))

# ------------------------------
# Helpers
# ------------------------------
def _infer_single(audio_array, sampling_rate):
    inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device)
    with torch.no_grad():
        logits = model(input_values).logits
    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0]
    return pred_text, logits.squeeze(0).cpu().numpy()

def _align(ref_tokens, hyp_tokens):
    # DP alignment for edit operations
    n, m = len(ref_tokens), len(hyp_tokens)
    dp = np.zeros((n + 1, m + 1), dtype=int)
    for i in range(n + 1):
        dp[i, 0] = i
    for j in range(m + 1):
        dp[0, j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if ref_tokens[i - 1] == hyp_tokens[j - 1] else 1
            dp[i, j] = min(
                dp[i - 1, j] + 1,
                dp[i, j - 1] + 1,
                dp[i - 1, j - 1] + cost,
            )
    # Backtrace
    aligned = []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            cost = 0 if ref_tokens[i - 1] == hyp_tokens[j - 1] else 1
            if dp[i, j] == dp[i - 1, j - 1] + cost:
                op = "equal" if cost == 0 else "substitute"
                aligned.append((ref_tokens[i - 1], hyp_tokens[j - 1], op))
                i -= 1
                j -= 1
                continue
        if i > 0 and dp[i, j] == dp[i - 1, j] + 1:
            aligned.append((ref_tokens[i - 1], None, "delete"))
            i -= 1
        else:
            aligned.append((None, hyp_tokens[j - 1], "insert"))
            j -= 1
    return list(reversed(aligned)), dp

def _normalize_devanagari(text):
    # Keep only Devanagari and spaces
    import re as _re
    text = _re.sub(r"[^\u0900-\u097F ]", "", text)
    text = _re.sub(r"\s+", " ", text).strip()
    return text

def _add_noise_snr(audio, snr_db):
    if snr_db == "clean":
        return audio
    rms = np.sqrt(np.mean(audio ** 2)) + 1e-8
    noise = np.random.normal(0.0, 1.0, size=audio.shape)
    noise_rms = np.sqrt(np.mean(noise ** 2)) + 1e-8
    desired_noise_rms = rms / (10 ** (snr_db / 20.0))
    noise = noise * (desired_noise_rms / noise_rms)
    return audio + noise

# ------------------------------
# Run evaluation on subset
# ------------------------------
records = []
for idx in range(len(val_ds)):
    row = val_ds[idx]
    audio = row["speech"]
    sr = row["sampling_rate"]
    ref = row["text"]
    start = time.perf_counter()
    hyp, logits = _infer_single(audio, sr)
    duration = len(audio) / float(sr)
    elapsed = time.perf_counter() - start
    rtf = elapsed / max(duration, 1e-6)
    records.append({
        "idx": idx,
        "ref": ref,
        "hyp": hyp,
        "wer": wer(ref, hyp),
        "rtf": rtf,
        "logits": logits,
        "duration": duration,
    })
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(val_ds)}")

df = pd.DataFrame(records)
print(df[["wer", "rtf"]].describe())

# ------------------------------
# 1) Levenshtein alignment matrix (example: worst WER)
# ------------------------------
worst = df.sort_values("wer", ascending=False).iloc[0]
ref_words = worst["ref"].split()
hyp_words = worst["hyp"].split()
_, dp_matrix = _align(ref_words, hyp_words)
plt.figure(figsize=(8, 6))
sns.heatmap(dp_matrix, cmap="mako", cbar=True)
plt.title("Levenshtein Alignment Matrix (Words)")
plt.xlabel("Hypothesis tokens")
plt.ylabel("Reference tokens")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "levenshtein_matrix_words.png"), dpi=200)
plt.show()

# ------------------------------
# 2) Confusion matrices (word + character)
# ------------------------------
# Build alignments
word_pairs = []
char_pairs = []
for _, row in df.iterrows():
    ref_norm = _normalize_devanagari(row["ref"])
    hyp_norm = _normalize_devanagari(row["hyp"])
    ref_words = ref_norm.split()
    hyp_words = hyp_norm.split()
    aligned_words, _ = _align(ref_words, hyp_words)
    for r, h, _op in aligned_words:
        word_pairs.append((r if r is not None else "<eps>", h if h is not None else "<eps>"))
    ref_chars = list(ref_norm.replace(" ", ""))
    hyp_chars = list(hyp_norm.replace(" ", ""))
    aligned_chars, _ = _align(ref_chars, hyp_chars)
    for r, h, _op in aligned_chars:
        char_pairs.append((r if r is not None else "<eps>", h if h is not None else "<eps>"))

# Word-level confusion (top-k words)
word_counts = pd.Series([r for r, _ in word_pairs]).value_counts()
top_words = list(word_counts.head(TOP_WORDS).index)
labels_words = top_words + ["<eps>", "<other>"]
def _map_word(w):
    if w in top_words or w == "<eps>":
        return w
    return "<other>"

word_y = [_map_word(r) for r, _ in word_pairs]
word_pred = [_map_word(h) for _, h in word_pairs]
cm_words = confusion_matrix(word_y, word_pred, labels=labels_words)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_words, cmap="rocket", xticklabels=labels_words, yticklabels=labels_words)
plt.title("Word Confusion Matrix (Top Words)")
plt.xlabel("Predicted")
plt.ylabel("Reference")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "confusion_words.png"), dpi=200)
plt.show()

# Character-level confusion (Devanagari + eps)
char_labels = sorted(set([c for c, _ in char_pairs] + [c for _, c in char_pairs]))
if len(char_labels) > 60:
    # Keep most frequent chars to keep plot readable
    char_counts = pd.Series([r for r, _ in char_pairs]).value_counts()
    char_labels = list(char_counts.head(50).index) + ["<eps>", "<other>"]
    def _map_char(c):
        if c in char_labels or c == "<eps>":
            return c
        return "<other>"
    char_y = [_map_char(r) for r, _ in char_pairs]
    char_pred = [_map_char(h) for _, h in char_pairs]
    cm_chars = confusion_matrix(char_y, char_pred, labels=char_labels)
else:
    char_y = [r for r, _ in char_pairs]
    char_pred = [h for _, h in char_pairs]
    cm_chars = confusion_matrix(char_y, char_pred, labels=char_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_chars, cmap="viridis", xticklabels=char_labels, yticklabels=char_labels)
plt.title("Character Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Reference")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "confusion_chars.png"), dpi=200)
plt.show()

# ------------------------------
# 3) Loss and metric curves (from trainer or train_output.txt)
# ------------------------------
def _extract_log_history():
    if "trainer" in globals() and hasattr(trainer, "state"):
        return trainer.state.log_history
    return None

def _parse_train_output(path="train_output.txt"):
    if not os.path.exists(path):
        return []
    logs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if "loss" in line or "eval_" in line:
                # Best-effort parse for JSON-like logs
                try:
                    if line.startswith("{") and line.endswith("}"):
                        logs.append(json.loads(line))
                except Exception:
                    continue
    return logs

history = _extract_log_history()
if not history:
    history = _parse_train_output()

if history:
    hist_df = pd.DataFrame(history)
    fig, ax1 = plt.subplots(figsize=(9, 5))
    if "loss" in hist_df:
        ax1.plot(hist_df["loss"].dropna().reset_index(drop=True), label="train_loss")
    if "eval_loss" in hist_df:
        ax1.plot(hist_df["eval_loss"].dropna().reset_index(drop=True), label="eval_loss")
    ax1.set_xlabel("Log step")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax2 = ax1.twinx()
    if "eval_wer" in hist_df:
        ax2.plot(hist_df["eval_wer"].dropna().reset_index(drop=True), color="red", label="eval_wer")
        ax2.set_ylabel("WER")
    plt.title("Loss and WER Curves")
    plt.tight_layout()
    plt.savefig(os.path.join(REPORT_DIR, "loss_wer_curves.png"), dpi=200)
    plt.show()
else:
    print("No training log history found for loss/WER curves.")

# ------------------------------
# 4) RTF vs Accuracy scatter
# ------------------------------
plt.figure(figsize=(7, 5))
plt.scatter(df["rtf"], df["wer"] * 100.0, alpha=0.7)
plt.xlabel("Real-Time Factor (RTF)")
plt.ylabel("WER (%)")
plt.title("RTF vs Accuracy")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "rtf_vs_wer.png"), dpi=200)
plt.show()

# ------------------------------
# 5) CTC logit alignment heatmap (approx)
# ------------------------------
sample = df.iloc[0]
logits = sample["logits"]
ref_text = _normalize_devanagari(sample["ref"])
ref_ids = processor.tokenizer(ref_text).input_ids
if ref_ids:
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    token_probs = probs[:, ref_ids]
    plt.figure(figsize=(10, 4))
    sns.heatmap(token_probs.T, cmap="icefire", cbar=True)
    plt.title("CTC Logit Alignment Heatmap (Ref Tokens)")
    plt.xlabel("Time frames")
    plt.ylabel("Reference token index")
    plt.tight_layout()
    plt.savefig(os.path.join(REPORT_DIR, "ctc_alignment_heatmap.png"), dpi=200)
    plt.show()
else:
    print("Skipping alignment heatmap: empty reference tokens.")

# ------------------------------
# 6) Robustness bar chart (noise vs WER)
# ------------------------------
robust_records = []
for snr in SNR_LEVELS_DB:
    wers = []
    for idx in range(len(val_ds)):
        row = val_ds[idx]
        audio = row["speech"]
        sr = row["sampling_rate"]
        noisy = _add_noise_snr(audio, snr)
        hyp, _ = _infer_single(noisy, sr)
        wers.append(wer(row["text"], hyp))
    robust_records.append({"snr": snr, "wer": float(np.mean(wers))})
robust_df = pd.DataFrame(robust_records)
plt.figure(figsize=(7, 4))
sns.barplot(data=robust_df, x="snr", y="wer")
plt.title("Noise Robustness (WER vs SNR)")
plt.xlabel("SNR (dB)")
plt.ylabel("WER")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "robustness_wer_snr.png"), dpi=200)
plt.show()

print("Saved figures to:", REPORT_DIR)

C:\Users\kadus\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\kadus\miniconda3\envs\sanskrit_asr_v2\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\kadus\miniconda3\envs\sanskrit_asr_v2\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\kadus\miniconda3\envs\sanskrit_asr_v2\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please 

RuntimeError: Dataset not found. Run the dataset loading/prep cells first.